<a href="https://colab.research.google.com/github/elisacapilli/ai-notebooks/blob/upload-notebooks/1_stats_Mistral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U langchain transformers bitsandbytes accelerate

!pip install langchain-community

import torch
from transformers import BitsAndBytesConfig
from langchain import HuggingFacePipeline
from langchain import PromptTemplate, LLMChain
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

from huggingface_hub import login
login(token="HF_TOKEN")

model_fp16 = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.1",
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
pipeline_inst = pipeline(
    "text-generation",
    model=model_fp16,
    tokenizer=tokenizer,
    use_cache=True,
    device_map="auto",
    max_length=2500,
    do_sample=True,
    top_k=5,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

llm = HuggingFacePipeline(pipeline=pipeline_inst)

import pandas as pd
import random
from IPython.display import display
df = pd.read_excel("df_facct.xlsx")

In [ ]:
value_counts = df['statement_text'].value_counts()
frequent_statements = value_counts[value_counts >= 5].index
df_filtered = df[df['statement_text'].isin(frequent_statements)]

In [ ]:
def remap_truthfulness_scale(df, col='truthfulness_value'):
    scale_map = {
        -2: 0,
        -1: 1,
        0: 2,
        1: 3,
        2: 5
    }
    df[col] = df[col].map(scale_map)
    return df
df_filtered = remap_truthfulness_scale(df_filtered)

In [ ]:
selected_statement = df_filtered['statement_text'].sample(1).iloc[0]
df_single_statement = df_filtered[df_filtered['statement_text'] == selected_statement]
print(f"Statement selezionato:\n{selected_statement}\n")
display(df_single_statement)

In [ ]:
df_filtered = df_single_statement.dropna(subset=['truthfulness_value'])
selected_rows = df_filtered.sample(n=5, random_state=42)
display(selected_rows)
truthfulness_values_list = selected_rows['truthfulness_value'].astype(int).tolist()
print(truthfulness_values_list)

In [ ]:
prompt = """You are a professional fact-checking AI. You will be presented with a statement along with five evaluations provided by anonymous human workers through crowdsourcing.
These workers assess the factual accuracy of the statement using the PolitiFact scale.

Each worker assigns an evaluation to the statement, and your task is to evaluate the truthfulness of the statement using real-world facts and verified evidence.
While you should consider the workers' evaluations, keep in mind that they may be inaccurate, biased, or may not fully reflect the true factuality of the statement.

PolitiFact Factuality Scale (integer values only):
0 = Pants on Fire — statement completely false and absurd
1 = False — statement entirely incorrect
2 = Mostly False — statement contains significant falsehoods or misleading elements
3 = Half True — statement mixes truth and falsehood
4 = Mostly True — statement mostly accurate with minor errors
5 = True — statement fully accurate and supported by verifiable facts

Your task:
- Evaluate the truthfulness of the statement based on factual, verified information.
- Consider the evaluations provided by the workers: {worker_votes_str}.
- Apply critical judgment, considering that the evaluations are provided in sequence.
- Based on the evaluations, provide a single integer answer from 0 to 5, representing your factuality rating for the statement.
- Your answer should consider all the five evaluations.
- Provide no explanations or additional text.

Statement: "{statement}"
Worker Evaluations: {worker_votes_str}
Answer:
"""

In [ ]:
def generate_prompt(row, worker_votes):
    worker_votes_str = "\n".join([f"- Worker {i+1}: {v}" for i, v in enumerate(worker_votes)])
    return prompt.format(statement=row['statement_text'], worker_votes_str=worker_votes_str)

prompt = PromptTemplate(template=prompt, input_variables=["statement", "worker_votes_str"])
llm_chain = LLMChain(prompt=prompt, llm=llm)

In [ ]:
import re
selected_row = selected_rows.iloc[0]
def get_model_response():
    inputs = {
        "statement": selected_row['statement_text'],
        "worker_votes_str": "\n".join([f"- Worker {i+1}: {v}" for i, v in enumerate(truthfulness_values_list)])
    }
    return llm_chain.run(inputs).strip()
def filter_model_response(response):
    matches = re.findall(r"\b[0-5]\b", response)
    if matches:
        return int(matches[-1])
    else:
        return "Invalid"
model_response = get_model_response()
filtered_response = filter_model_response(model_response)

results_df = pd.DataFrame([{
    "Statement": selected_row['statement_text'],
    "Worker Evaluations": truthfulness_values_list,
    "Ground Truth": selected_row['fact_check_ground_truth_value'],
    "Model Response": filtered_response,
}])
print("Risposta del modello:", model_response)
display(results_df)